# Scrape FAQ Help Center

Notebook ini mengambil data FAQ dari `http://13.229.198.150/help-center`, memecah tiap item menjadi **question** dan **answer**, lalu menyimpan hasilnya ke JSON.

In [2]:
import ast
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import requests
from bs4 import BeautifulSoup

URL = "http://13.229.198.150/help-center"
OUT_PATH = Path("help_center_faq.json")

resp = requests.get(URL, timeout=30)
resp.raise_for_status()
html = resp.text
soup = BeautifulSoup(html, "html.parser")


def parse_js_string_literal(token: str) -> str:
    token = token.strip()
    try:
        value = ast.literal_eval(token)
        return value if isinstance(value, str) else str(value)
    except Exception:
        if len(token) >= 2 and token[0] == token[-1] and token[0] in {'"', "'"}:
            return token[1:-1]
        return token


def extract_field_from_js_object(body: str, key: str) -> str:
    pattern = re.compile(
        rf"\b{re.escape(key)}\s*:\s*(?P<value>'(?:\\.|[^'\\])*'|\"(?:\\.|[^\"\\])*\")",
        flags=re.DOTALL,
    )
    match = pattern.search(body)
    if not match:
        return ""
    return parse_js_string_literal(match.group("value")).strip()


def parse_faq_from_js(soup_obj: BeautifulSoup) -> list[dict]:
    scripts_text = "\n".join((sc.string or sc.get_text() or "") for sc in soup_obj.find_all("script"))

    push_pattern = re.compile(
        r"faqData\.category_faqs(?P<path>(?:\[[^\]]+\])+?)\.push\(\{(?P<body>.*?)\}\);",
        flags=re.DOTALL,
    )
    path_key_pattern = re.compile(r"\[['\"]([^'\"]+)['\"]\]")

    rows = []
    for match in push_pattern.finditer(scripts_text):
        path_expr = match.group("path")
        body = match.group("body")

        keys = path_key_pattern.findall(path_expr)
        category = keys[0] if len(keys) >= 1 else ""
        subcategory = keys[1] if len(keys) >= 2 else ""
        subsubcategory = keys[2] if len(keys) >= 3 else ""

        faq_id = extract_field_from_js_object(body, "id")
        question = extract_field_from_js_object(body, "question")
        answer = extract_field_from_js_object(body, "answer")

        if not question and not answer:
            continue

        rows.append(
            {
                "faq_id": faq_id,
                "question": question,
                "answer": answer,
                "category": category,
                "subcategory": subcategory,
                "subsubcategory": subsubcategory,
                "source": "inline_js",
            }
        )

    return rows


def parse_faq_from_dom(soup_obj: BeautifulSoup) -> list[dict]:
    rows = []
    for item in soup_obj.select(".faq-item"):
        question_node = item.select_one(".faq-question")
        answer_node = item.select_one(".faq-answer")

        question = question_node.get_text(" ", strip=True) if question_node else ""
        answer = answer_node.get_text(" ", strip=True) if answer_node else ""

        if not question and not answer:
            continue

        rows.append(
            {
                "faq_id": (item.get("data-faq-id") or "").strip(),
                "question": question,
                "answer": answer,
                "category": (item.get("data-category") or "").strip(),
                "subcategory": (item.get("data-subcategory") or "").strip(),
                "subsubcategory": (item.get("data-subsubcategory") or "").strip(),
                "source": "dom",
            }
        )
    return rows


faq_rows_raw = parse_faq_from_js(soup)
if not faq_rows_raw:
    faq_rows_raw = parse_faq_from_dom(soup)

faq_rows = []
seen = set()
for row in faq_rows_raw:
    key = (row.get("faq_id", ""), row.get("question", ""), row.get("answer", ""))
    if key in seen:
        continue
    seen.add(key)
    faq_rows.append(row)

sidebar_points = []
for link in soup.select(".category-list a"):
    text = link.get_text(" ", strip=True)
    href = (link.get("href") or "").strip()
    if text:
        sidebar_points.append({"label": text, "href": href})

result = {
    "source_url": URL,
    "scraped_at_utc": datetime.now(timezone.utc).isoformat(),
    "total_faq": len(faq_rows),
    "representation": "JSON with one object per FAQ: faq_id, question, answer, category hierarchy",
    "faq": faq_rows,
    "sidebar_points": sidebar_points,
}

OUT_PATH.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Saved: {OUT_PATH.resolve()}")
print(f"Total FAQ extracted: {len(faq_rows)}")
print(f"Sidebar points found: {len(sidebar_points)}")

if faq_rows:
    print("\nSample FAQ:")
    print(json.dumps(faq_rows[0], ensure_ascii=False, indent=2))

Saved: C:\Documents\Kuliah\Skripsi\in-app-navigational-agent\scrape_faq\help_center_faq.json
Total FAQ extracted: 121
Sidebar points found: 15

Sample FAQ:
{
  "faq_id": "pengaturan_umum_1",
  "question": "Profil Perusahaan",
  "answer": "Profil Perusahaan digunakan untuk menyimpan dan memperbarui informasi identitas perusahaan yang terdaftar di sistem Fingerspot.io. Pastikan seluruh data sesuai dengan perusahaan saat ini, karena informasi ini berfungsi sebagai data utama yang tersimpan dalam sistem Fingerspot.io dan tampilan profil perusahaan di sistem.\n\n                    Kelengkapan informasi profil perusahaan, diantaranya:\n                    • Nama Perusahaan\n                    • Alamat\n                    • Telepon\n                    • Email\n                    • Nama Kontak\n                    • Ponsel Kontak\n                    • Zona Waktu\n                    • Unggah Logo Perusahaan di Web\n                    • Unggah Logo Perusahaan di App Fingerspot.io",
  "ca

<unknown>:1: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\/" is an i